# Main collection

Collect n unique subjects after `r/power.R` has locked sample size
from the largest of the eight pilot cell SDs (Granite scope in
`docs/framework-empirical-evaluation-granite.md`; protocol Section 6).

Do **not** pool the pilot into the eight primary TOSTs unless the design
is later amended. After this notebook, render `r/analysis.qmd` on the
main `evaluation.csv` and `ledger.csv`.

Needs CUDA.

**Google Colab:** Runtime → Change runtime type → GPU (T4 or L4). Run the
bootstrap cell first. Put `sample_size.json` from the pilot (or set `N`
by hand) under the Drive artifacts folder before collecting.

In [ ]:
# Colab / local bootstrap.
# Colab: Runtime → Change runtime type → GPU, then run this cell.
# Local: skip clone and Drive if the repo is already on disk.

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/RobinGirardin/zepto.git"
STUDY_BRANCH = "empirical-test"
DRIVE_STUDY_DIR = Path("MyDrive") / "zepto-granite-validity"


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        marker = candidate / "studies" / "granite-validity" / "python"
        if marker.is_dir():
            return candidate
    return None


IN_COLAB = in_colab()
REPO_ROOT = find_repo_root(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ARTIFACTS_ROOT = Path("/content/drive") / DRIVE_STUDY_DIR
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
else:
    ARTIFACTS_ROOT = None

if REPO_ROOT is None:
    if not IN_COLAB:
        raise FileNotFoundError(
            "could not find studies/granite-validity; open this notebook from the repo"
        )
    clone_dir = Path("/content/zepto")
    if not (clone_dir / "studies" / "granite-validity" / "python").is_dir():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                STUDY_BRANCH,
                "--depth",
                "1",
                REPO_URL,
                str(clone_dir),
            ]
        )
    REPO_ROOT = clone_dir

STUDY_ROOT = REPO_ROOT / "studies" / "granite-validity"
if ARTIFACTS_ROOT is None:
    ARTIFACTS_ROOT = STUDY_ROOT / "artifacts"

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "studies" / "validity_common" / "python"))
sys.path.insert(0, str(STUDY_ROOT / "python"))

if IN_COLAB:
    # Do not pip-install torch: Colab already has a CUDA build.
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"]
    )

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "this notebook needs a CUDA GPU. In Colab: Runtime → Change runtime type → GPU."
    )

print("colab:", IN_COLAB)
print("repo:", REPO_ROOT)
print("study:", STUDY_ROOT)
print("artifacts:", ARTIFACTS_ROOT)
print("gpu:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
import json

from granite_validity import Catalog, run_collection

size_path = ARTIFACTS_ROOT / "pilot" / "sample_size.json"
if size_path.exists():
    N = int(json.loads(size_path.read_text())["n_subjects"])
    print("n from power.R:", N)
else:
    N = None
    print("sample_size.json not found; set N by hand")

SEED = 2
OUT = ARTIFACTS_ROOT / "main"
catalog = Catalog()
N, OUT

In [ ]:
if N is None:
    raise ValueError("set N from power.R before collecting the main sample")

run_collection(N, seed=SEED, output_dir=OUT, catalog=catalog)
print("wrote", OUT)
print("next: quarto render studies/granite-validity/r/analysis.qmd \\")
print("  -P evaluation:../artifacts/main/evaluation.csv \\")
print("  -P ledger:../artifacts/main/ledger.csv")